## Data Collection — Library Setup

In this section, the required libraries for data collection are loaded. These libraries are used for making API requests, parsing web content, and handling data.

- `httr` is used to send HTTP requests to APIs
- `jsonlite` is used to parse JSON responses
- `rvest` is used for web scraping

In [1]:
# Load required libraries for data collection
library(httr)        # Used to send API requests
library(jsonlite)    # Used to parse JSON responses
library(rvest)       # Used for web scraping HTML content

## Load Primary Dataset

The Seoul bike-sharing dataset is loaded as the primary data source. This dataset contains historical information on bike rentals along with weather and temporal variables.

In [2]:
# Load Seoul bike-sharing dataset
bike_data <- read.csv("seoul_bike_sharing.csv")  # Read CSV file

# Display first few rows
head(bike_data)  # Inspect dataset

# Check structure
str(bike_data)  # Understand data types

,DATE,RENTED_BIKE_COUNT,HOUR,TEMPERATURE,HUMIDITY,WIND_SPEED,VISIBILITY,DEW_POINT_TEMPERATURE,SOLAR_RADIATION,RAINFALL,SNOWFALL,SEASONS,HOLIDAY,FUNCTIONING_DAY
,<chr>,<int>,<int>,<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
1,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0,0,0,Winter,No Holiday,Yes
2,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0,0,0,Winter,No Holiday,Yes
3,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0,0,0,Winter,No Holiday,Yes
4,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0,0,0,Winter,No Holiday,Yes
5,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0,0,0,Winter,No Holiday,Yes
6,01/12/2017,100,5,-6.4,37,1.5,2000,-18.7,0,0,0,Winter,No Holiday,Yes


'data.frame':	8465 obs. of  14 variables:
 $ DATE                 : chr  "01/12/2017" "01/12/2017" "01/12/2017" "01/12/2017" ...
 $ RENTED_BIKE_COUNT    : int  254 204 173 107 78 100 181 460 930 490 ...
 $ HOUR                 : int  0 1 2 3 4 5 6 7 8 9 ...
 $ TEMPERATURE          : num  -5.2 -5.5 -6 -6.2 -6 -6.4 -6.6 -7.4 -7.6 -6.5 ...
 $ HUMIDITY             : int  37 38 39 40 36 37 35 38 37 27 ...
 $ WIND_SPEED           : num  2.2 0.8 1 0.9 2.3 1.5 1.3 0.9 1.1 0.5 ...
 $ VISIBILITY           : int  2000 2000 2000 2000 2000 2000 2000 2000 2000 1928 ...
 $ DEW_POINT_TEMPERATURE: num  -17.6 -17.6 -17.7 -17.6 -18.6 -18.7 -19.5 -19.3 -19.8 -22.4 ...
 $ SOLAR_RADIATION      : num  0 0 0 0 0 0 0 0 0.01 0.23 ...
 $ RAINFALL             : num  0 0 0 0 0 0 0 0 0 0 ...
 $ SNOWFALL             : num  0 0 0 0 0 0 0 0 0 0 ...
 $ SEASONS              : chr  "Winter" "Winter" "Winter" "Winter" ...
 $ HOLIDAY              : chr  "No Holiday" "No Holiday" "No Holiday" "No Holiday" ...
 $ FUNCTIONING

## Weather Data Collection using OpenWeather API

Weather data is collected using the OpenWeather API to capture environmental conditions such as temperature, humidity, and wind speed. These variables are important for understanding their impact on bike rental demand. a232e7c2307255216c9ed2580ca72984

In [3]:
# Define API key and city
api_key <- "a232e7c2307255216c9ed2580ca72984"   # Replace with valid API key
city <- "Seoul"             # Specify city name

# Construct API URL
url <- paste0(
  "https://api.openweathermap.org/data/2.5/forecast?q=",
  city,
  "&appid=",
  api_key,
  "&units=metric"
)

# Send GET request to API
response <- GET(url)  # Request weather data

# Convert response to text
content_data <- content(response, "text")  # Extract raw response

# Parse JSON response
weather_json <- fromJSON(content_data)  # Convert JSON to R object

# Inspect structure
names(weather_json)  # View available fields

[1] "cod"     "message" "cnt"     "list"    "city"

## Extract Relevant Weather Features

We extract key variables from the API response for further analysis.

In [4]:
# --------------------------------------------
# Extract weather data (flattened JSON case)
# --------------------------------------------

# Extract the list (which is actually a dataframe)
weather_list <- weather_json$list  # Already structured as dataframe

# Create clean weather dataset
weather_data <- data.frame(
  
  # Extract datetime directly
  datetime = weather_list$dt_txt,  # Timestamp column
  
  # Extract temperature from nested 'main' dataframe
  temperature = weather_list$main$temp,  # Temperature values
  
  # Extract humidity from nested 'main'
  humidity = weather_list$main$humidity,  # Humidity values
  
  # Extract wind speed from nested 'wind'
  wind_speed = weather_list$wind$speed,  # Wind speed values
  
  # Extract weather condition from nested list
  weather = sapply(weather_list$weather, function(x) x$main)  # Weather type
)

# Preview dataset
head(weather_data)

,datetime,temperature,humidity,wind_speed,weather
,<chr>,<dbl>,<int>,<dbl>,<chr>
1,2026-04-26 09:00:00,20.76,23,3.93,Clear
2,2026-04-26 12:00:00,19.19,22,1.96,Clear
3,2026-04-26 15:00:00,18.43,23,1.46,Clouds
4,2026-04-26 18:00:00,15.66,29,1.13,Clouds
5,2026-04-26 21:00:00,14.76,33,1.05,Clouds
6,2026-04-27 00:00:00,16.91,26,0.46,Clouds


## Final Data Preparation for Downstream Processing

In this section, the collected datasets are finalized and stored in a structured format. This ensures that all data sources are ready for data wrangling, exploratory data analysis, and modeling in subsequent stages of the project.

The datasets prepared include:
- Weather data collected from the OpenWeather API
- Bike-sharing system data collected via web scraping

These datasets will be used in the next stage for data cleaning and transformation.

In [5]:
# Standardize column names
colnames(weather_data) <- tolower(colnames(weather_data))

# Save weather dataset
write.csv(weather_data, "weather_data.csv", row.names = FALSE)

# Confirm file creation
file.exists("weather_data.csv")

[1] TRUE

In [6]:
summary(weather_data)

   datetime          temperature       humidity       wind_speed   
 Length:40          Min.   :10.87   Min.   :13.00   Min.   :0.460  
 Class :character   1st Qu.:13.67   1st Qu.:25.75   1st Qu.:1.625  
 Mode  :character   Median :15.79   Median :38.00   Median :2.860  
                    Mean   :16.37   Mean   :39.58   Mean   :3.168  
                    3rd Qu.:18.62   3rd Qu.:50.25   3rd Qu.:4.515  
                    Max.   :23.04   Max.   :82.00   Max.   :6.820  
   weather         
 Length:40         
 Class :character  
 Mode  :character  
                   
                   
                   

## Data Validation Summary

The summary statistics indicate that the collected weather dataset is complete and consistent. No missing values are observed across the variables, and all numerical features fall within reasonable ranges. 

These results confirm that the dataset is suitable for further preprocessing and integration with the bike-sharing data in subsequent stages.

---

## Author & Acknowledgment

**Author:**  
<span style="color:blue">Deepan Mehta  </span>

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook focuses on collecting data from multiple sources including APIs, web scraping, and cloud-based datasets.

The workflow is based on <span style="color:blue"> IBM Skills Network </span> instructional labs on data collection techniques for real-world data science projects.

Special acknowledgment is given to:

- <span style="color:blue">Yan Luo  </span>
- <span style="color:blue">Jeff Grossman  </span>

---

## Project Context

This notebook represents the data collection stage in the end-to-end data science pipeline:

- Data Collection (API, Web Scraping, CSV)
- Data Wrangling (ETL)
- Exploratory Data Analysis (EDA)
- Model Development
- Model Evaluation
- Deployment (R Shiny Dashboard)

---

## Notes

All code and explanations have been independently rewritten to ensure clarity, reproducibility, and alignment with the capstone project requirements.

---